# 🕵️ Dia 60 — Regras de detecção com Suricata

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 60: Detection rules with Suricata* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-60.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Você escreveu `alert ip $HOME_NET any -> 198.51.100.140 any` em dois minutos. Rafa: *"Essa regra morre na quinta-feira. Me escreve uma que ainda funcione quando o IP mudar. E testa: regra que você não viu disparar é desejo, não detecção."*

Neste notebook você vai:

1. **Checar a sintaxe** das regras com um analisador próprio (o seu `suricata -T` de bolso).
2. **Rodar um mini-motor de regras** sobre pacotes sintéticos e gerar um `eve.json`.
3. **Reproduzir o passo 6**: o *threshold* do *spray* do LAB-P4 contra o log do firewall.
4. **Montar a tabela de alertas** do LAB-P4 e da Isca, com "não testada em pacotes" onde cabe.
5. **Jogar o quiz "indicador ou comportamento?"**.

🐍 Só biblioteca padrão. O mini-motor entende só um subconjunto (`flags:S`, destino/porta, `threshold`); o Suricata de verdade fica na célula opcional.

## 1. Carregando os firewalls

In [ ]:
# ------------------------------------------------------------
# CARREGADOR OFFLINE-FIRST
# Tenta baixar cada arquivo do GitHub (raw). Se a rede falhar,
# usa a cópia EMBUTIDA aqui embaixo (comprimida para caber no notebook).
# Nos dois casos, calculamos o SHA-256: é a nossa "cadeia de custódia" de bolso.
# ------------------------------------------------------------
import base64          # para decodificar as cópias embutidas
import hashlib         # SHA-256 de cada arquivo carregado
import urllib.request  # download simples, sem bibliotecas externas
import zlib            # as cópias embutidas estão comprimidas com zlib

# Endereço "cru" dos dados do LAB-P4 no repositório original (ThiefDroppers)
RAW_LAB = ("https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/"
           "curriculum/phase-4-digital-forensics-incident-investigation/resources/case-lab-p4/")
# Endereço "cru" dos dados sintéticos da Operação Isca neste repositório
RAW_BR = ("https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/"
          "caso-continuo/dados/")

# Plano B: cópias fiéis dos arquivos, feitas a partir do repositório local.
EMBUTIDOS = {
    "lab:logs/firewall.log": """
eNqt291qXDcQB/D7PMW+wC6ab+lACsXpdU1JH8AhDgkYu2Q3hLx9tc42tY4d/7XxgMlV7N9KGs3M0VlxYd8W2ZJsyJcSSymbD18L
bX6/uPjj8u3m7cXlhsqu/2jZWSzaWuXN9rcNNd6VHe+4LKqy+fzl5vr11c3N3dftx8Phn/327sth8+7b4Xr/Ws21veIBovZzyPq/
vFgpOkAEIaJmvoK0QKi1Jg8hmYBUaQWZ4qlz8gFyBEVVlzVUn4WcFjOqA8QQEtE6OLGwIEdFw4Ylqsjh2lqsoYBLVCvTMHMEoRJq
K6jPNxqRxhjdOBZCi60hZTSivo1iGJFCiIN1cOpC2PEqet52leiLtIYUxlxtZYgFVjigHttryGAsKEkbgpscQ/3DDVBbJOCIgric
F3SVYpUXWt8kMLqjthimLuAa0ZhR+6oSTN3dqXUYUMPbtVpZQYx3UfgI4SVS6yl1BU3sIqZVLMCZi2J1HBF15Qf05q8/L78zre6M
OlV20WPBezq5h+517h9gEantu/T++sPVl5vD9v317bcT448MftbwHvpk9kID79LWdFwZgStDEraGHFdv8jGRMs471WQFPZfgTkXV
Y6xBgvNOUaprKJ5dnurHjH2qQb+6PAyMvjzMp3j+VUNgX9XLlQ9pjWEWUG1jWuuQEgwBqirn5U/XXujXkMF0Y1zKeSFgscqffe4J
1+waet7mYVFex7TBzaPVx1oqBjvSvneG1pcZFe1jRSh9jc6D3IJtBSmGqseYqHHpIeayhhoekfGQ4NjwGgmtoWMw8COIi/S/SSS7
HpTauNTVXmV+CO33H7fvrvaHT3e3P3IpPyGRQok4RWLBkqdIgiXWFKnXMyRJSZB6JcJjkkiRKKCkliJxw1JLkQSvk3GKZISljCjv
jT2WXFMkMizVFInxOnlLkRTPXuSMyQqWMsakM1WjRoo0kSOap0gC10mKpEgKZ0+IUiSbkDJmzyaqhjClSDhHSEol7L+lWIoUSRxK
KimSNixljMknqoZoTZEIx57ljInxOllLkXDVEM+ZPcOzFxnVPfB5Wpif98herY4H+ico8JA8ZUiMlyml5MbEo4ZUSZEUp72U4n58
XoZSy3gAqAsVLNUUiWF50pQ2oi4Co1yppEj44VNTGpY20VgqtRQJFw1lTpEYr1NKw9ImGksVTZFw0VB5cYaV+1cWOPa0pEj42Ehf
3oTdSzIxpkiRFMeeZcweTbR7ajVFItiEqVOKhA8JNHJmbyIiImf28COh1oydyxNdmDZKkRiPqbUUSWB1t6IpEj7MMcqYPZmoufby
g/l7iSakSJFwzTXWFEkalixFMtiXm2SMSfsPllqKhHO5pVRCXQTvJ/MUSXGOSKlP2hsxLGXEnk10y+Y1RcK53MJTJHzgaxEpkuLZ
q5wiTeSImrFzvWNQapIiMd65raZI+IzFC6dIuC//77sRL5UMZlinjKoRHcNSpEj4pZCn1NyYeHHsIikSfgLwlCfqmOgsPeWJ+skv
cD+SIkfC66QtRSK8TkYpEu5h3SxFwi8b3HLWCZ8becopwcxJoltLkXBv5K4pEn4h6Sn9Xpt4IekhKdJENqqUI+EckdEbUZnJRs1T
JJyNokiKJHA/RcYpwfH6k2MpY0wzZ5bBJUWiCSlSJPxGMjJOCbqEz8JCOUXC759CM8Y08+XHUEuR8FlYZPQRNHM6GiYpEq4aYTmz
N5EjMqo7yUQPG15SJPwF6XBPkfCbrghNkSZyRM2IPYVPn1aIM7JReyrDPvibSsMqSTne1noMfbo9XH++vbr5cbGhri6GycM7vT+7
S8Xt0RvJcy7SyBP3W/5+czl8dJPhPXhXjk38g9G8v92vR9P4CYZ/MmvfGWl2Oob4f4SMb21VJmllPXP2/My10v9HvOgKktx/5xHc
b7F66ienb4OICbdX/wLJa96W""",
    "br:dia-57/fw-serraazul.log": """
eNqtmdtqHEcQhu/9FPsCu9S5uwYcCCL3vvALyKAQgxKHSEb47V2zjhJ1zbKlaQv2Slrpmzr9dRgCsiPCEfQAbQGIz+H3p+PD7eHX
m5vfPnw8fLz5cEA4EZwEToqLKroejr8c0OkUPz8hLiJ8+Ofr/d372/v7L0/HPx4f/344fvn6ePj07fHu4b00g/6ORpLF5xqJFhUw
f0lir0gKBpZICAvSNRIv2tlsILWK1AgcMkkK79Ei3tD3kRRFs/cofM7XSLqoSzzMS5JVJOPwQybFf7pqU1tJNNhEUnqvk3km+UJS
xIm740CiikTNuSUSS00K5402QUmi+EIm9YVa4T1mkKGeuCJxY8REkr4AFjaZYtsXJwKRHKfQAKk0oiPyvsolB8/eiywWLzQC0Ics
lzJOJj3lXrhOF5CCZM6DRlBZT82wtUSK6JIWGdFgrNyowcomR7RM6ou0QiO4oe3T8mYdKJM8Er2wCQH7D1KPDDkhrL/EpUOGvWT5
BZJgSSJaSQQcNkUinlqpseJukFihsaAli3EvqwnjFqVUo2w3itrGKi664RklsNuD0aU2iRGJCyUMn5P9v8TAuojRXTLMCljkO7kM
Wku1C5WkZ5JHf6jNogmzumbBiGZ1XQQjdwy1D3JbCkYodGpWZ5JaaRbifrOIeo4WV5Uc/cpBRnWqFZeUL4BaaRVN5CCB5uqSeGq9
Hixx9Z1WIUrLkiERLKzN0hmzSDMsOmSvgkVOQ2mVY4yRWE4L8XrcDLN8wizeyK6GD72EMc3AMIvGeVAuR4w2+BDL8ZYjxFkx1F/R
TZCnfAhjeXmMTYuUXTLyd0Y0iDJs3eYqGLcJGGN0L4TMC+2oPakyIfSNNcFie2SttkcT2rf/mBFsQVKnvbaZrpzm3YC1coaPnRh0
nyIKcOuZFGlfTqFoOCX0nmD0Kul4PirsS3vMaR+TDVe3Eupt5w7OrJLNYliAa7P8Tao5YEzFZMO9DQs/l6AOlnOQ8eJk+P946+v8
3exHHZ/xemHhf7r79Oft5/vj57+eaws3tRUo1AoVje9tUJf28AElTd8ERRdXoeRA+nc0fD2qd9kke8yu9dqFbUqdNGsuW3016a0P
2/i6VxeVZeJZneLxoFanPtFKom1hhr1KnRxn+pZlH2pYVg7YNLPkEUvfwgiLJS9WNt43icYcuun9+po8jAqYCtgwReF6dF+Qa9jU
yNYhw7hYX3WxGIZk3/GYYiXfkDS6cmkW8s8vyuuRPz51HqLPCMeY9CtJF6oDRvzzl42AhZJSqR3EM5MNjofkFYYLt+o43o13vsQg
aTlgpPVxPMyaGjbGy+EK64VZeL7ZDGZhebVmZ8tJv441tXDIzGgIZhmmxQx1humM0jtn2PpHUMNmKqxzzsOyh60vCJVo3zmPQTmX
l3ph1vrqBNpwYSOvTTKFd98BEId8CA==""",
}

REGISTRO_DE_CARGA = []   # (arquivo, origem, sha256): vai para as suas notas de método


def carregar(chave):
    """chave = 'lab:logs/firewall.log' ou 'br:dia-57/proxy-sa.log'. Devolve o texto."""
    prefixo, rel = chave.split(":", 1)                    # separa de onde vem e o caminho
    url = (RAW_LAB if prefixo == "lab" else RAW_BR) + rel  # monta a URL raw
    reserva = zlib.decompress(base64.b64decode(EMBUTIDOS[chave].replace("\n", "")))  # bytes da cópia embutida
    try:
        with urllib.request.urlopen(url, timeout=10) as r:  # tenta a rede, com tempo limite
            dados, origem = r.read(), "rede"
    except Exception:                                       # sem rede, 404, timeout: tanto faz
        dados, origem = reserva, "embutido"
    if origem == "rede" and dados != reserva:              # a versão online mudou desde que copiamos?
        print(f"⚠️  {rel}: versão online difere da embutida; usando a embutida (números do dia batem com ela)")
        dados, origem = reserva, "embutido (online divergente)"
    sha = hashlib.sha256(dados).hexdigest()                # impressão digital do que vamos analisar
    REGISTRO_DE_CARGA.append((rel, origem, sha))           # anota para o relatório
    return dados.decode("utf-8")


print(f"{len(EMBUTIDOS)} arquivos disponíveis. Use carregar('...').")

In [ ]:
FW_LAB = carregar("lab:logs/firewall.log").splitlines()
FW_SA = carregar("br:dia-57/fw-serraazul.log").splitlines()
print(len(FW_LAB), len(FW_SA))

In [ ]:
# Tabela de carga: o que foi lido, de onde e com qual hash.
for rel, origem, sha in REGISTRO_DE_CARGA:     # uma linha por arquivo
    print(f"{rel:40s} {origem:10s} {sha[:16]}…")  # 16 primeiros caracteres bastam para conferir de olho

## 2. O arquivo `local.rules` e um `suricata -T` de bolso

In [ ]:
LOCAL_RULES = r"""
# Detecta: POST HTTP para example.com. Teste: captura própria do Dia 59.
alert http $HOME_NET any -> $EXTERNAL_NET any (msg:"LAB own-capture HTTP POST to example.com"; flow:established,to_server; http.method; content:"POST"; http.host; content:"example.com"; sid:9000001; rev:1;)
# Detecta: consulta DNS ao nome NXDOMAIN do laboratório. Teste: captura própria.
alert dns $HOME_NET any -> any any (msg:"LAB own-capture DNS query for lab NXDOMAIN name"; dns.query; content:"nonexistent-lab-name"; nocase; sid:9000002; rev:1;)
# Detecta: SNI example.org. Teste: captura própria.
alert tls $HOME_NET any -> $EXTERNAL_NET any (msg:"LAB own-capture TLS SNI example.org"; tls.sni; content:"example.org"; sid:9000003; rev:1;)
# Detecta: 3 GETs em 180 s para example.com. Teste: captura própria.
alert http $HOME_NET any -> $EXTERNAL_NET any (msg:"LAB own-capture repeated GET to example.com"; flow:established,to_server; http.method; content:"GET"; http.host; content:"example.com"; threshold: type threshold, track by_src, count 3, seconds 180; sid:9000004; rev:1;)
# Detecta: rajada de SYN para SSH. NÃO testada em pacotes (falta pcap do LAB-P4); replay no log do firewall.
alert tcp $EXTERNAL_NET any -> $HOME_NET 22 (msg:"LAB-P4 SSH connection burst from one source"; flow:to_server; flags:S,12; threshold: type threshold, track by_src, count 20, seconds 300; sid:9000010; rev:1;)
# Detecta: qualquer tráfego para o indicador. NÃO testada em pacotes.
alert ip $HOME_NET any -> 198.51.100.23 any (msg:"LAB-P4 traffic to case indicator 198.51.100.23"; sid:9000011; rev:1;)
# Isca: indicador (vai envelhecer). Teste: pcap sintético.
alert tcp $HOME_NET any -> 198.51.100.140 443 (msg:"ISCA SA-CX-07 conexao ao indicador atualiza-modulo"; flow:to_server; flags:S,12; sid:9000020; rev:1;)
# Isca: comportamento, HTTPS repetido. Teste: pcap sintético.
alert tcp $HOME_NET any -> $EXTERNAL_NET 443 (msg:"ISCA conexoes HTTPS repetidas de uma origem"; flow:to_server; flags:S,12; threshold: type threshold, track by_src, count 10, seconds 3600; sid:9000021; rev:1;)
# Isca: SNI imitação. NÃO testada em pacotes (pcap sintético sem TLS).
alert tls $HOME_NET any -> $EXTERNAL_NET any (msg:"ISCA SNI imitando serraazul"; tls.sni; content:"serraazul"; nocase; sid:9000022; rev:1;)
# Regra com erro de propósito (falta o ; depois de rev:1) — o checador deve reclamar:
alert tcp any any -> any 23 (msg:"TESTE erro proposital"; sid:9000099; rev:1)
"""
import re

CABECALHO = re.compile(r"^(alert|pass|drop|reject)\s+(\w+)\s+(\S+)\s+(\S+)\s+(->|<>)\s+(\S+)\s+(\S+)\s+\((.*)\)$")

def checar(regras_txt):
    """Devolve lista de regras válidas (dicts) e imprime erros, como um 'suricata -T' bem simples."""
    ok = []
    for n, linha in enumerate(regras_txt.strip().splitlines(), 1):
        linha = linha.strip()
        if not linha or linha.startswith("#"):
            continue                                           # comentário: é documentação
        m = CABECALHO.match(linha)
        if not m:
            print(f"❌ linha {n}: cabeçalho inválido"); continue
        acao, proto, src, sport, dir_, dst, dport, opcoes = m.groups()
        if not opcoes.rstrip().endswith(";"):
            print(f"❌ linha {n}: a última opção não termina com ';'  ({opcoes[-25:]})"); continue
        ops = [o.strip() for o in re.split(r";(?=(?:[^\"]*\"[^\"]*\")*[^\"]*$)", opcoes) if o.strip()]
        d = {"acao": acao, "proto": proto, "src": src, "sport": sport, "dst": dst, "dport": dport, "ops": ops}
        d["sid"] = int(next(o.split(":")[1] for o in ops if o.startswith("sid:")))
        d["msg"] = next(o.split(":", 1)[1].strip('"') for o in ops if o.startswith("msg:"))
        if d["sid"] < 9000000:
            print(f"⚠️  linha {n}: sid {d['sid']} fora da faixa local (use 9000000+)")
        ok.append(d)
    print(f"{len(ok)} regras carregadas")
    return ok

REGRAS = checar(LOCAL_RULES)

> 💡 O Suricata real acusa muito mais (palavra-chave desconhecida, *sticky buffer* fora do lugar). O nosso só pega o erro mais comum: o `;` faltando. Apague a última regra (a de teste) do seu `local.rules` real.

## 3. Um mini-motor: pacotes sintéticos → `eve.json`

Construímos pacotes SYN a partir dos fluxos do firewall da Serra Azul (como no Dia 59) e aplicamos só o que o nosso motor entende: protocolo `tcp`/`ip`, destino, porta, `flags:S` e `threshold`. Regras de `http`, `dns` e `tls` precisam de **conteúdo de aplicação**, que o pcap sintético não tem: o motor diz "sessão não interpretada", exatamente a pergunta do diagrama do original.

In [ ]:
import ipaddress, json
from datetime import datetime, timedelta, timezone
from collections import defaultdict, deque

UTC = timezone.utc
try:
    from zoneinfo import ZoneInfo
    SP, NY = ZoneInfo("America/Sao_Paulo"), ZoneInfo("America/New_York")
except Exception:
    SP, NY = timezone(timedelta(hours=-3)), timezone(timedelta(hours=-4))

HOME_NET = [ipaddress.ip_network(n) for n in ("10.0.0.0/8", "172.16.0.0/12", "192.168.0.0/16")]  # o --set do passo 3

def eh_home(ip):
    return any(ipaddress.ip_address(ip) in n for n in HOME_NET)

def parse_fw(linha, fuso):
    p = linha.split()
    dt = datetime.fromisoformat(f"{p[0]} {p[1]}").replace(tzinfo=fuso).astimezone(UTC)
    s, sp = p[5].rsplit(":", 1); d, dp = p[7].rsplit(":", 1)
    return {"ts": dt, "proto": p[4].lower(), "src": s, "sport": int(sp), "dst": d, "dport": int(dp),
            "acao_fw": p[3], "bytes": int(p[9].split("=")[1]), "syn": True}   # 1 fluxo = 1 SYN

def casa_endereco(campo, ip):
    if campo == "any": return True
    if campo == "$HOME_NET": return eh_home(ip)
    if campo == "$EXTERNAL_NET": return not eh_home(ip)
    return ipaddress.ip_address(ip) in ipaddress.ip_network(campo)

def casa_porta(campo, porta):
    return campo == "any" or int(campo) == porta

def rodar(regras, pacotes):
    eve, contagem = [], defaultdict(int)
    janelas = defaultdict(deque)                                        # (sid, origem) -> horários recentes
    for r in regras:
        if r["proto"] not in ("tcp", "ip"):
            eve.append({"event_type": "stats", "nota": f"sid {r['sid']}: proto {r['proto']} precisa de conteúdo de aplicação; sessão não interpretada"})
    for p in sorted(pacotes, key=lambda p: p["ts"]):
        for r in regras:
            if r["proto"] not in ("tcp", "ip"): continue
            if r["proto"] == "tcp" and p["proto"] != "tcp": continue
            if not (casa_endereco(r["src"], p["src"]) and casa_endereco(r["dst"], p["dst"])): continue
            if not (casa_porta(r["sport"], p["sport"]) and casa_porta(r["dport"], p["dport"])): continue
            if any(o.startswith("flags:S") for o in r["ops"]) and not p["syn"]: continue
            th = next((o for o in r["ops"] if o.startswith("threshold")), None)
            if th:                                                     # type threshold: alerta a cada N em T s
                cnt = int(re.search(r"count (\d+)", th).group(1)); seg = int(re.search(r"seconds (\d+)", th).group(1))
                jan = janelas[(r["sid"], p["src"])]
                jan.append(p["ts"])
                while (p["ts"] - jan[0]).total_seconds() > seg:
                    jan.popleft()
                if len(jan) < cnt: continue
                jan.clear()                                            # 'threshold' reinicia a contagem após alertar
            contagem[r["sid"]] += 1
            eve.append({"timestamp": p["ts"].isoformat(), "event_type": "alert", "src_ip": p["src"],
                        "dest_ip": p["dst"], "dest_port": p["dport"],
                        "alert": {"signature_id": r["sid"], "signature": r["msg"]}})
    return eve, contagem

pacotes_sa = [parse_fw(l, SP) for l in FW_SA]
eve_sa, cont_sa = rodar(REGRAS, pacotes_sa)
with open("eve.json", "w", encoding="utf-8") as f:
    for e in eve_sa:
        f.write(json.dumps(e, ensure_ascii=False) + "\n")
print({sid: n for sid, n in sorted(cont_sa.items())})

In [ ]:
# O "jq" do passo 4, em Python: alertas agrupados por assinatura.
from collections import Counter
alertas = Counter()
with open("eve.json", encoding="utf-8") as f:
    for linha in f:
        ev = json.loads(linha)
        if ev["event_type"] == "alert":
            alertas[(ev["alert"]["signature_id"], ev["alert"]["signature"])] += 1
        else:
            print("ℹ️ ", ev["nota"])
print()
for (sid, sig), n in sorted(alertas.items()):
    print(f"{n:4d}  {sid}  {sig}")

Repare na **9000021**: ela dispara para o `SA-CX-07` (o *beacon*), mas olhe de quem mais é o alerta, se houver. Agora baixe o `count` na célula abaixo (5, 3, 2) e veja quando os caixas vizinhos começam a entrar. No dado sintético a rotina dos vizinhos é esparsa; numa agência real, com navegadores abertos o dia todo, eles entrariam bem antes. É o custo em falsos positivos.

In [ ]:
#@title Ajuste o threshold da 9000021 { display-mode: "form" }
novo_count = 5  #@param {type:"integer"}
novo_seconds = 3600  #@param {type:"integer"}

r21 = next(r for r in REGRAS if r["sid"] == 9000021)
copia = dict(r21, ops=[o if not o.startswith("threshold") else
                       f"threshold: type threshold, track by_src, count {novo_count}, seconds {novo_seconds}"
                       for o in r21["ops"]])
eve_t, _ = rodar([copia], pacotes_sa)
por_origem = Counter(e["src_ip"] for e in eve_t if e["event_type"] == "alert")
print(f"count {novo_count} / {novo_seconds} s → alertas por origem: {dict(por_origem)}")

## 4. Passo 6: o *threshold* do *spray* no log do firewall do LAB-P4

In [ ]:
pacotes_lab = [parse_fw(l, NY) for l in FW_LAB]
spray = [p for p in pacotes_lab if p["src"] == "203.0.113.45"]

# Intervalos de 5 minutos em hora LOCAL do firewall (como o awk do original)
baldes = Counter()
for p in spray:
    local = p["ts"].astimezone(NY)
    baldes[local.strftime("%Y-%m-%d %H:") + f"{local.minute // 5 * 5:02d}"] += 1
for k in sorted(baldes):
    print(k, baldes[k])
print(f"total: {len(spray)} conexões")

In [ ]:
# Janela deslizante (o que o Suricata faz de verdade) para vários thresholds
def primeiro_alerta(eventos, count, seconds):
    jan = deque()
    for t in eventos:
        jan.append(t)
        while (t - jan[0]).total_seconds() > seconds:
            jan.popleft()
        if len(jan) >= count:
            return t
    return None

ts_spray = sorted(p["ts"] for p in spray)
for count in (10, 20, 25, 30):
    t = primeiro_alerta(ts_spray, count, 300)
    print(f"count {count:2d} / 300 s → " + (f"dispara às {t:%H:%M:%S}Z" if t else "NUNCA dispara"))
print("\nE se o mesmo spray fosse mais lento? (count 20 / 300 s)")
def pico(eventos, seconds):
    """Maior número de eventos dentro de qualquer janela de 'seconds' segundos."""
    jan, maior = deque(), 0
    for t in eventos:
        jan.append(t)
        while (t - jan[0]).total_seconds() > seconds:
            jan.popleft()
        maior = max(maior, len(jan))
    return maior

for fator in (1.0, 1.5, 2.0, 2.5):
    lento = [ts_spray[0] + (t - ts_spray[0]) * fator for t in ts_spray]
    t = primeiro_alerta(lento, 20, 300)
    print(f"  ritmo x{fator:.1f}: pico de {pico(lento, 300)} conexões em 300 s → "
          + (f"dispara às {t:%H:%M:%S}Z" if t else "NUNCA dispara"))
print("Conclusão: count 20 / 300 s pega este spray, mas um operador paciente passa por baixo.")

## 5. A tabela de alertas

In [ ]:
TABELA = [
    # sid, esperado, disparou, contagem, notas
    (9000001, "sim", "—", "—", "captura própria (Dia 59); rode o Suricata real"),
    (9000002, "sim", "—", "—", "captura própria; pode disparar 2x se o resolvedor repetir"),
    (9000003, "sim", "—", "—", "captura própria"),
    (9000004, "sim", "—", "—", "captura própria; dispara no 3º GET em 180 s"),
    (9000010, "sim", "não testada em pacotes", "—", "replay no firewall: dispara por pouco com count 20; testar em <data>"),
    (9000011, "sim", "não testada em pacotes", "—", "indicador; testar em <data>; preferir regra por volume"),
    (9000020, "sim", "sim" if cont_sa.get(9000020) else "não", cont_sa.get(9000020, 0), "pcap sintético; indicador, vai envelhecer"),
    (9000021, "sim", "sim" if cont_sa.get(9000021) else "não", cont_sa.get(9000021, 0), "pcap sintético; comportamento, ruidosa com count baixo"),
    (9000022, "sim", "não testada em pacotes", "—", "pcap sintético sem TLS; exceção para serraazul.example"),
]
print("| sid | Esperado disparar | Disparou | Contagem | Notas |\n|---|---|---|---|---|")
for linha in TABELA:
    print("| " + " | ".join(map(str, linha)) + " |")

### ✍️ O esboço do passo 7 (regra por volume)

Escreva com as suas palavras no formulário. Exemplo: *"Alertar quando um host de `$HOME_NET` enviar mais de 20 MB para um único endereço de `$EXTERNAL_NET` numa só sessão TLS (campo `flow.bytes_toserver` do `eve.json`), com exceção dos destinos de backup conhecidos."*

In [ ]:
#@title Minha regra por volume { display-mode: "form" }
esboco = "Alertar quando um host interno enviar mais de 20 MB para um único endereço externo numa só sessão TLS (flow.bytes_toserver), exceto destinos de backup conhecidos."  #@param {type:"string"}
from pathlib import Path
Path("notas").mkdir(exist_ok=True)
Path("notas/regra-por-volume.md").write_text("# Regra por volume (passo 7)\n\n" + esboco + "\n", encoding="utf-8")
Path("local.rules").write_text(LOCAL_RULES, encoding="utf-8")
print("💾 notas/regra-por-volume.md, local.rules, eve.json")

## 6. 🎮 Quiz: indicador ou comportamento?

Checkpoint BR: pelo menos 5 de 6.

In [ ]:
# ------------------------------------------------------------
# QUIZ: função genérica de múltipla escolha
# MODO_JOGO = True  -> pergunta e conta acertos
# MODO_JOGO = False -> só mostra gabarito e explicação
# ------------------------------------------------------------
def jogar_quiz(perguntas, modo_jogo=True, minimo=None):
    acertos = 0                                            # placar
    for n, (enunciado, opcoes, certa, porque) in enumerate(perguntas, start=1):
        print(f"\n[{n}/{len(perguntas)}] {enunciado}")    # o enunciado
        for letra, texto in opcoes.items():                # as alternativas a), b), c)
            print(f"   {letra}) {texto}")
        if modo_jogo:
            resposta = ""
            for _ in range(5):                             # até 5 tentativas de digitar uma letra válida
                resposta = input("   Sua resposta: ").strip().lower()
                if resposta in opcoes:
                    break
            ok = (resposta == certa)                       # acertou?
            acertos += ok
            print("   ✅ Acertou!" if ok else f"   ❌ Era {certa}).")
        else:
            print(f"   Gabarito: {certa})")
        print(f"   Por quê: {porque}")
    if modo_jogo:
        print(f"\n🏁 Resultado: {acertos}/{len(perguntas)}")
        if minimo is not None:
            print("Checkpoint BR cumprido. 🥷" if acertos >= minimo
                  else f"Checkpoint BR pede pelo menos {minimo}. Releia o README e tente de novo.")
    return acertos

In [ ]:
MODO_JOGO = True
PERGUNTAS = [
    ("`alert ip $HOME_NET any -> 198.51.100.140 any` é uma regra de:",
     {"a": "indicador (envelhece quando o IP muda)", "b": "comportamento", "c": "anomalia estatística"},
     "a", "Funciona até o kit trocar de servidor, o que na Isca já aconteceu quatro vezes."),
    ("'Host interno envia mais de 20 MB a um destino externo numa sessão' é:",
     {"a": "indicador", "b": "comportamento", "c": "assinatura de malware"},
     "b", "Vale para qualquer IP de destino."),
    ("`suricata -T` passou sem erro. A regra está pronta?",
     {"a": "sim", "b": "não: falta vê-la disparar onde você esperava, em tráfego que entende", "c": "só se o sid for 9000000+"},
     "b", "Sintaxe é a primeira checagem do ciclo, não a última."),
    ("Sem `-k none`, numa captura feita no próprio host, as regras podem:",
     {"a": "disparar em dobro", "b": "deixar de disparar em silêncio por causa de checksum", "c": "corromper o pcap"},
     "b", "A placa calcula o checksum depois da captura (offload)."),
    ("A 9000022 (SNI contém 'serraazul') também casa com:",
     {"a": "nada além do golpe", "b": "o site legítimo serraazul.example", "c": "só tráfego DNS"},
     "b", "Daí a exceção para o domínio legítimo."),
    ("Uma regra 99% precisa numa rede com 50 mil conexões/dia e 1 ataque/ano gera:",
     {"a": "quase só alertas verdadeiros", "b": "centenas de falsos positivos por verdadeiro", "c": "zero falsos positivos"},
     "b", "Negligência da taxa-base. E o analista cansado fecha o verdadeiro junto."),
]
jogar_quiz(PERGUNTAS, MODO_JOGO, minimo=5)

## 7. (Opcional) O Suricata de verdade, na sua VM

In [ ]:
!sudo apt-get -y install suricata jq
!suricata -V
!sudo suricata -T -c /etc/suricata/suricata.yaml -S local.rules -v
print("Rode na sua VM de laboratório contra a sua captura do Dia 59 e contra o pcap sintético.")

In [ ]:
try:
    from google.colab import files
    for f in ("local.rules", "eve.json", "notas/regra-por-volume.md"):
        files.download(f)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta atual.)")

## ✅ Antes de fechar o Dia 60

- [ ] `local.rules` com uma linha de comentário acima de cada regra
- [ ] Tabela de alertas com 9 linhas e "não testada em pacotes" onde cabe
- [ ] Esboço da regra por volume
- [ ] Quiz com pelo menos 5/6

**Amanhã (Dia 61):** triagem estática. O `msa-atualizador.exe` não vai ser executado; vai ser descrito.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>